# AI Model Experiment & Evaluation
**Student:** Ahnaf Rifqi

Assignment: membandingkan **model klasik Machine Learning (Scikit-learn)** dan **LLM API (Gemini AI)** untuk task **sentiment analysis** ulasan pelanggan e-commerce, lalu menyusun rekomendasi teknis berbasis data eksperimen.

Alur: **Prepare** (memahami studi kasus & menyiapkan dataset) → **Experiment** (latih model klasik + inference LLM pada data yang sama) → **Evaluate** (Accuracy, Precision, Recall, F1-Score, confusion matrix) → **Recommend** (analisis trade-off & rekomendasi).


> **Catatan Dataset**
> Dataset: `data/customer_reviews_sentiment.csv` — kolom `review_text` (teks) dan `sentiment` (label: `positif`/`negatif`), 200 baris. Dataset memiliki banyak teks duplikat (40 teks unik), sehingga split dikelompokkan berdasarkan teks untuk mencegah kebocoran duplikat. Test set yang sama digunakan untuk **kedua** pendekatan.

## 1. Problem Statement

**Objective:** Mengklasifikasikan ulasan pelanggan e-commerce secara otomatis sebagai sentimen positif atau negatif agar tim produk dapat mengevaluasi pengalaman pelanggan dan menindaklanjuti ulasan bermasalah.

**Target/Label:** Kolom `sentiment` dengan dua kelas: `positif` dan `negatif`.

**Batasan/Asumsi:** Dataset berisi 200 ulasan berbahasa Indonesia; label yang tersedia dianggap sebagai ground truth. Banyak baris mengulang teks yang sama, sehingga split acak per baris berisiko membocorkan contoh identik. Eksperimen menggunakan `StratifiedGroupKFold` berdasarkan teks ulasan: teks unik tidak muncul di train dan test sekaligus. Kedua pendekatan memakai test set yang sama (40 baris, 8 teks unik). Ukuran test yang kecil membatasi generalisasi hasil.

## 2. Import Library & Load Dataset


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from google import genai
from google.genai import types

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

# Muat variabel lingkungan dari .env (API key tidak di-hardcode)
load_dotenv()

# Basis path: root project, baik saat notebook dijalankan dari root maupun dari folder notebook/
ROOT = Path.cwd()
if ROOT.name == 'notebook':
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'customer_reviews_sentiment.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset tidak ditemukan di {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nDistribusi label:')
print(df['sentiment'].value_counts())
df.head()


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Grup berdasarkan teks agar duplikat ulasan tidak bocor ke train dan test.
LABELS = ['negatif', 'positif']
X = df['review_text']
y = df['sentiment']
groups = df['review_text']

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Split yang sama untuk kedua pendekatan; setiap teks unik hanya ada di satu sisi.
test_ids = df.iloc[test_idx]['review_id'].reset_index(drop=True)
print('Train size:', len(X_train))
print('Test size :', len(X_test))
print('Teks unik di test:', X_test.nunique())
print('Teks overlap train/test:', len(set(X_train) & set(X_test)))
print('\nDistribusi kelas di test set:')
print(y_test.value_counts())

## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction


In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print('Jumlah fitur TF-IDF:', X_train_vec.shape[1])


### 4.2 Training Model


In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
print('Model terlatih. Kelas:', clf.classes_)


### 4.3 Prediksi pada Test Set


In [ ]:
y_pred_classic = clf.predict(X_test_vec)
print('Contoh prediksi (10 pertama):', y_pred_classic[:10])

# Simpan hasil prediksi model klasik (output wajib ada di repository)
predictions_classic = pd.DataFrame({
    'review_id': test_ids,
    'review_text': X_test.reset_index(drop=True),
    'actual_sentiment': y_test.reset_index(drop=True),
    'pred_classic': y_pred_classic,
})
predictions_classic.to_csv(ROOT / 'data' / 'predictions_classic.csv', index=False)
predictions_classic.head()


## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API


In [ ]:
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
MODEL_NAME = 'gemini-3.8-flash'
client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
if client is None:
    print('GEMINI_API_KEY tidak tersedia. Bagian LLM akan dilewati; model klasik tetap dapat dievaluasi.')
else:
    print('Gemini client siap. Model:', MODEL_NAME)

### 5.2 Merancang Prompt

Menggunakan pendekatan **zero-shot**: model diminta mengklasifikasikan sentimen tanpa contoh label.

- Instruksi eksplisit mengenai output yang diharapkan (hanya satu kata: `positif`/`negatif`).
- `temperature=0.2` dipilih agar output lebih konsisten untuk tugas klasifikasi (semakin rendah, semakin deterministik).
- Output dinormalisasi agar sesuai label dataset (`positif`/`negatif`).


In [ ]:
def normalize_llm_output(raw_text):
    """Normalisasi output LLM agar konsisten dengan label 'positif'/'negatif'."""
    if raw_text is None:
        return 'unknown'
    match = re.search(r'\b(positif|negatif)\b', raw_text.strip().lower())
    return match.group(1) if match else 'unknown'


def classify_sentiment_llm(review_text):
    """Klasifikasi sentimen satu ulasan menggunakan Gemini API (zero-shot)."""
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return normalize_llm_output(response.text)


### 5.3 Menjalankan Prediksi pada Test Set

Prediksi dijalankan pada **test set yang sama** dengan model klasik. Karena test set hanya berisi 8 teks unik (dari 40 baris), API dipanggil **sekali per teks unik**, lalu hasilnya dipetakan ke semua baris dengan teks yang sama — hemat kuota & biaya ~5x dibanding memanggil per baris. Dilengkapi *cache* (`data/llm_predictions.csv`) agar menjalankan ulang notebook tidak memanggil API berulang kali; cache otomatis melanjutkan dari teks yang belum selesai (resume), jadi aman dijalankan bertahap saat kuota harian.

In [ ]:
import time

CACHE_PATH = ROOT / 'data' / 'llm_predictions.csv'

if CACHE_PATH.exists():
    cache_df = pd.read_csv(CACHE_PATH)
    print(f'Memeriksa {len(cache_df)} prediksi cache dari {CACHE_PATH}')
else:
    cache_df = pd.DataFrame(columns=['review_id', 'review_text', 'pred_llm'])

# Cache dibaca dari ID+teks. Prediksi ditentukan per TEKS UNIK: test set berisi 40
# baris tetapi hanya 8 teks unik, jadi API dipanggil sekali per teks unik (hemat
# kuota & biaya hingga ~5x), lalu hasilnya dipetakan ke semua baris dengan teks sama.
row_cache = {}
for row in cache_df.itertuples(index=False):
    try:
        row_cache[int(row.review_id)] = (str(row.review_text), normalize_llm_output(row.pred_llm))
    except (AttributeError, TypeError, ValueError):
        continue

text_pred = {}
for rid, review in zip(test_ids, X_test):
    cached = row_cache.get(int(rid))
    if cached and cached[0] == str(review) and cached[1] in LABELS:
        text_pred.setdefault(str(review), cached[1])

unique_reviews = X_test.unique().tolist()
print('Teks unik di test:', len(unique_reviews))

# Inferensi dengan retry + melanjutkan ke teks berikutnya saat satu permintaan gagal
# (mis. kuota harian habis), sehingga prediksi yang berhasil tetap tersimpan.
llm_errors = {}
MAX_ATTEMPTS = 3
for i, review in enumerate(unique_reviews, start=1):
    if str(review) in text_pred:
        continue
    if client is None:
        llm_errors[str(review)] = 'GEMINI_API_KEY tidak tersedia.'
        text_pred[str(review)] = 'unknown'
        continue
    pred = 'unknown'
    for attempt in range(MAX_ATTEMPTS):
        try:
            pred = classify_sentiment_llm(review)
            break
        except Exception as exc:
            llm_errors[str(review)] = f'{type(exc).__name__}: {exc}'
            print(f'  [{i}/{len(unique_reviews)}] teks gagal percobaan {attempt + 1}: {str(exc)[:200]}', flush=True)
            if attempt < MAX_ATTEMPTS - 1:
                time.sleep(5 * (attempt + 1))
    text_pred[str(review)] = pred
    print(f'Diproses {i}/{len(unique_reviews)} teks unik', flush=True)

y_pred_llm = [text_pred[str(r)] for r in X_test]

# Simpan hanya prediksi yang valid; yang masih 'unknown' akan dicoba ulang di eksekusi berikutnya.
valid_cache = pd.DataFrame([
    {'review_id': rid, 'review_text': review, 'pred_llm': pred}
    for rid, review, pred in zip(test_ids, X_test, y_pred_llm)
    if pred in LABELS
])
valid_cache.to_csv(CACHE_PATH, index=False)

predictions_llm = pd.DataFrame({
    'review_id': test_ids,
    'review_text': X_test.reset_index(drop=True),
    'actual_sentiment': y_test.reset_index(drop=True),
    'pred_llm': y_pred_llm,
})
predictions_llm.to_csv(ROOT / 'data' / 'predictions_llm.csv', index=False)
print(f'Prediksi LLM tersedia: {sum(p in LABELS for p in y_pred_llm)}/{len(X_test)}')
if llm_errors:
    first = next(iter(llm_errors.values()))
    print('Ada permintaan yang gagal, contoh:', first[:300])
    print('Jalankan ulang sel ini setelah kuota harian reset; cache otomatis melanjutkan sisanya.')
predictions_llm.head()

## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik


In [ ]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_classic, labels=LABELS))


Pada split bebas kebocoran, model memperoleh accuracy **0,625**, precision macro **0,6278**, recall macro **0,6288**, dan F1 macro **0,6248**. Confusion matrix (baris aktual, kolom prediksi; urutan `negatif`, `positif`) adalah `[[12, 6], [9, 13]]`. Model benar pada 12 dari 18 ulasan negatif dan 13 dari 22 ulasan positif. Kesalahan false-positive dan false-negative sama-sama masih sering terjadi; hasil ini jauh lebih realistis daripada split acak per baris yang menempatkan duplikat train/test dan menghasilkan skor 100%.

### 6.2 Evaluasi LLM API


In [ ]:
if len(y_pred_llm) == len(y_test) and all(pred in LABELS for pred in y_pred_llm):
    print('=== LLM API (Gemini) ===')
    print('Accuracy:', accuracy_score(y_test, y_pred_llm))
    print(classification_report(y_test, y_pred_llm, labels=LABELS, zero_division=0))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred_llm, labels=LABELS))
else:
    print(f'Evaluasi LLM belum valid: tersedia {len(y_pred_llm)} dari {len(y_test)} prediksi.')
    print('Lengkapi inference Gemini dan jalankan ulang bagian evaluasi setelah API tersedia.')

Jika seluruh prediksi tersedia, interpretasikan metrik dan confusion matrix Gemini yang tercetak di atas. Status saat ini: 3 dari 8 teks unik sudah terprediksi (ter-cache), sedangkan 5 teks sisanya menunggu kuota harian free-tier reset (HTTP 429). Setelah kuota tersedia, jalankan ulang sel 5.3 — cache otomatis memproses sisa teks — lalu sel evaluasi ini mencetak metrik Gemini. Tidak ada kesimpulan performa LLM yang dibuat dari cache parsial.

### 6.3 Tabel Perbandingan Ringkasan

_Metrik dihitung untuk kedua pendekatan pada test set yang sama (40 ulasan)._


In [ ]:
def get_metrics(y_true, y_pred):
    """Accuracy + Precision/Recall/F1 (macro & weighted) untuk perbandingan ringkas."""
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (macro)': precision_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0),
        'Recall (macro)': recall_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0),
        'F1-Score (macro)': f1_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0),
        'F1-Score (weighted)': f1_score(y_true, y_pred, average='weighted', labels=LABELS, zero_division=0),
    }

if len(y_pred_llm) == len(y_test) and all(pred in LABELS for pred in y_pred_llm):
    summary = pd.DataFrame({
        'Model Klasik (Scikit-learn)': get_metrics(y_test, y_pred_classic),
        'LLM API (Gemini)': get_metrics(y_test, y_pred_llm),
    }).round(4)
    summary.to_csv(ROOT / 'documentation' / 'comparison_summary.csv', index=True)
    display(summary)
else:
    summary = None
    print('Tabel perbandingan belum dibuat: hasil Gemini belum lengkap untuk test set yang sama.')

In [ ]:
if len(y_pred_llm) == len(y_test) and all(pred in LABELS for pred in y_pred_llm):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, (name, y_pred) in zip(axes, [
        ('Model Klasik', y_pred_classic),
        ('LLM API (Gemini)', y_pred_llm),
    ]):
        cm = confusion_matrix(y_test, y_pred, labels=LABELS)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=LABELS, yticklabels=LABELS)
        ax.set_title(name)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.tight_layout()
    (ROOT / 'documentation').mkdir(exist_ok=True)
    plt.savefig(ROOT / 'documentation' / 'model_comparison_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Confusion matrix tersimpan di documentation/model_comparison_summary.png')
else:
    print('Visualisasi perbandingan ditunda hingga prediksi Gemini lengkap tersedia.')

Model klasik mencapai accuracy **0,625** dan macro-F1 **0,6248** pada split yang memisahkan teks duplikat. Confusion matrix menunjukkan 15 kesalahan dari 40 baris, dengan kesalahan pada kedua kelas. Skor ini merupakan baseline pada hanya 8 teks unik test, bukan bukti kesiapan produksi.

Model klasik membutuhkan preprocessing, training, dan pemeliharaan vocabulary, tetapi inference lokal tidak menimbulkan biaya per-request dan memberi latensi yang dapat dikendalikan. Gemini tidak memerlukan training khusus dan mudah dicoba lewat prompt, tetapi setiap prediksi memerlukan API, menambah latensi jaringan, berpotensi menimbulkan biaya, dan bergantung pada kuota/layanan. Status saat ini 3/8 teks unik sudah terprediksi dan 5 lainnya menunggu kuota harian reset; performa kedua model belum dapat dibandingkan secara sah hingga seluruh 8 teks unik (40 baris) terprediksi.

Dataset sangat kecil dan repetitif (200 baris tetapi hanya 40 teks unik). Split acak per baris menghasilkan kebocoran: 39 dari 40 teks test juga muncul di train dan skor klasik 100%, sehingga pendekatan itu menyesatkan. Split grup menghindari kebocoran, tetapi hanya menguji 8 teks unik; kesalahan pada ungkapan yang belum pernah dilihat masih nyata. Label dataset juga diasumsikan benar dan belum ada pengujian pada data produksi.

Gunakan Logistic Regression + TF-IDF sebagai **baseline lokal** untuk iterasi berikutnya: implementasinya ringan, inference tidak bergantung pada kuota API, dan macro-F1 terukur **0,6248** pada split bebas kebocoran. Namun, performanya belum cukup untuk menyatakan siap produksi. Jangan menetapkan pemenang antara model klasik dan Gemini sebelum seluruh 8 teks unik (40 ulasan) test mendapat prediksi Gemini pada split yang sama. Setelah kuota tersedia, jalankan ulang bagian LLM dan evaluasi; kumpulkan lebih banyak teks ulasan unik, perbaiki kesalahan false-positive/false-negative, lalu validasi pada test set representatif yang lebih besar sebelum keputusan produksi.